# Compare the three saved chess models

The default comparison evaluates compact, reference and large checkpoints on
identical validation positions with one frozen analyzer. No training is run by
this notebook. Use `EXPERIMENT` for a grouped campaign, or `RUNS` for saved reports.

One seed per architecture and unequal training histories limit the conclusion
to these saved checkpoints. [Full report](../docs/report/2026-11-07-three-models.md).

Raw results are local and ignored by Git. For the portable measurements and
figures available after cloning, open the [report index](../docs/report/README.md).


## 1. Select an experiment

Set `EXPERIMENT` to the directory containing `config.json`. Leave it as `None`
only for the optional standalone `RUNS` comparison. Standalone reports have no
configuration/seed metadata and cannot be averaged into training configurations.


In [ ]:
from pathlib import Path
import sys
from html import escape
import matplotlib.pyplot as plt
from IPython.display import HTML, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.comparison import load_comparison, load_experiment, load_run_history

EXPERIMENT = None
COMPARISON_RUN = "results/comparisons/2026-11-07-three-models-01"
RUNS = [f"{COMPARISON_RUN}/{name}" for name in ("compact", "reference", "large")]
TRAINING_RUNS = {
    "compact": "results/training/2026-11-07-compact-blas-01",
    "reference": "results/training/2026-11-07-continue-reference-01",
    "large": "results/training/2026-11-07-large-blas-01",
}
VALIDATION_DATASET = ROOT / "dataset/references/chess-v1/splits/validation.txt"


## 2. Compare configurations across seeds

Scores show **mean ± sample standard deviation**; this is not a confidence interval.
A single seed has no estimated standard deviation. Missing seeds remain explicit.
Only a finished experiment with every planned seed and at least two seeds can
establish leading configurations. Default seeds and all settings come from the
experiment file, not this notebook. Exact mean F1 ties use mean worst-class recall.

Timing includes loading/startup and is descriptive. See the
[protocol](../docs/evaluation.md#comparison-protocol-v1).


In [ ]:
summary = load_experiment(EXPERIMENT) if EXPERIMENT is not None else None
if summary is not None:
    comparison = summary["runs"]
    print(f"Experiment: {EXPERIMENT} | Status: {summary['status']}")
    print("Planned seeds:", summary["expected_seeds"])
    if summary["selection_ready"]:
        print("Leading configurations on validation:", ", ".join(summary["leading_configurations"]))
    else:
        print("No configuration selection: experiment unfinished, missing seeds, or only one planned seed.")
else:
    comparison = (load_comparison([ROOT / name for name in RUNS],
                                  VALIDATION_DATASET) if RUNS else [])
    print("Standalone reports: no seed aggregation without an experiment plan.")

def table(headings, rows):
    header = "".join(f"<th>{escape(str(value))}</th>" for value in headings)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in row) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

def score(stat):
    if stat["mean"] is None:
        return "—"
    if stat["std"] is None:
        return f"{stat['mean']:.3f} (one seed)"
    return f"{stat['mean']:.3f} ± {stat['std']:.3f}"

if summary is not None:
    table(("Configuration", "Seeds completed", "Accuracy", "Macro F1", "Worst recall", "Train (s)", "Predict (s)", "Model (KiB)"),
          [(row["configuration"], f"{len(row['seeds'])}/{len(summary['expected_seeds'])}",
            score(row["accuracy"]), score(row["macro_f1"]), score(row["worst_recall"]),
            score(row["training_seconds"]), score(row["prediction_seconds"]),
            f"{row['model_bytes']['mean'] / 1024:.1f}" if row["model_bytes"]["mean"] is not None else "—")
           for row in summary["configurations"]])
    for row in summary["configurations"]:
        if row["missing_seeds"]:
            print(f"{row['configuration']}: missing seeds {row['missing_seeds']}")


In [ ]:
if summary is not None:
    groups = [row for row in summary["configurations"] if row["macro_f1"]["mean"] is not None]
    if groups:
        fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
        for index, row in enumerate(groups):
            stat = row["macro_f1"]
            ax.errorbar(index, stat["mean"], yerr=stat["std"] if stat["std"] is not None else None,
                        fmt="o" if row["complete"] else "x", capsize=5, color="tab:blue")
        ax.set_xticks(range(len(groups)), [row["configuration"] + ("" if row["complete"] else " (partial)") for row in groups])
        ax.set_xlim(-0.5, len(groups) - 0.5)
        ax.set_ylim(-0.05, 1.05)
        ax.set_ylabel("Validation macro F1")
        ax.set_title("Mean and sample standard deviation across training seeds")
        ax.spines[["top", "right"]].set_visible(False)
        plt.show()


## 3. Inspect individual runs

Compare validation scores, class recall and confusion matrices. Rows are actual
classes and columns are predictions. Counts and row percentages are shown;
colors share the same 0–100% scale. A single seed cannot estimate variability.


In [ ]:
if comparison:
    table(("Run", "Accuracy", "Macro F1", "Worst recall", "Elapsed (s)"),
          [(report["run"], f"{report['metrics']['accuracy']:.2%}",
            f"{report['metrics']['macro_f1']:.3f}", f"{report['worst_recall']:.3f}",
            f"{report['metrics']['prediction_seconds']:.2f}") for report in comparison])
    print("Validation SHA-256:", comparison[0]["metrics"]["artifacts"]["dataset"]["sha256"])
else:
    print("No completed evaluations yet. Prepare/train an experiment, or select standalone validation RUNS.")


In [ ]:
if comparison:
    labels = comparison[0]["metrics"]["confusion_matrix"]["labels"]
    fig, axes = plt.subplots(1, 2, figsize=(12, max(4, 0.45 * len(comparison) + 2)),
                             layout="constrained")
    names = [report["run"] for report in comparison]
    width = 0.24
    for ax, key in zip(axes, ("f1", "recall")):
        for offset, label in enumerate(labels):
            positions = [i + (offset - 1) * width for i in range(len(comparison))]
            values = [report["metrics"]["per_class"][label][key] for report in comparison]
            bars = ax.barh(positions, values, width, label=label)
            ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
        ax.set_yticks(range(len(names)), names)
        ax.invert_yaxis()
        ax.set_xlim(0, 1.15)
        ax.set_xlabel("Score")
        ax.set_title("F1 by class" if key == "f1" else "Recall by class")
        ax.spines[["top", "right"]].set_visible(False)
    axes[0].legend(loc="upper center", bbox_to_anchor=(0.5, 1.18), ncols=3)
    plt.show()


In [ ]:
if comparison:
    fig, axes = plt.subplots(1, len(comparison), figsize=(6 * len(comparison), 4),
                             squeeze=False, layout="constrained")
    for ax, report in zip(axes[0], comparison):
        matrix = report["metrics"]["confusion_matrix"]
        counts = matrix["counts"]
        fractions = [[value / sum(row) for value in row] for row in counts]
        plot = ax.imshow(fractions, vmin=0, vmax=1, cmap="Blues")
        for i, row in enumerate(counts):
            for j, count in enumerate(row):
                ax.text(j, i, f"{count:,}\n{fractions[i][j]:.1%}",
                        ha="center", va="center",
                        color="white" if fractions[i][j] > 0.5 else "black")
        ax.set_xticks(range(3), matrix["labels"])
        ax.set_yticks(range(3), matrix["labels"])
        ax.set(xlabel="Predicted", ylabel="Actual", title=report["run"])
    fig.colorbar(plot, ax=list(axes[0]), label="Fraction of actual class")
    plt.show()


## 4. Read training curves when available

A growing train/validation loss gap may indicate overfitting. Curves require the
backend's optional `history.csv`; missing curves are reported, never fabricated.

For the reference, the original 100 epochs and continuation are joined using
recorded metadata. Curves include epochs after the selected checkpoint.


In [ ]:
if comparison:
    histories = []
    for report in comparison:
        directory = (Path(report["directory"]) if summary is not None else
                     ROOT / TRAINING_RUNS.get(report["run"], report["directory"]))
        histories.append((report["run"], load_run_history(directory)))
    available = [(name, rows) for name, rows in histories if rows]
    if not available:
        print("No training curves yet. The backend may emit history.csv using the documented format.")
    for name, rows in available:
        fig, ax = plt.subplots(figsize=(7, 3), layout="constrained")
        ax.plot([row["epoch"] for row in rows], [row["train_loss"] for row in rows], label="Train")
        ax.plot([row["epoch"] for row in rows], [row["validation_loss"] for row in rows], label="Validation")
        ax.set(xlabel="Epoch", ylabel="Loss", title=name)
        ax.legend()
        ax.spines[["top", "right"]].set_visible(False)
        plt.show()
    for name, rows in histories:
        if not rows:
            print(f"{name}: no training history")


## Conclusions

The reference retains the highest validation macro F1: **81.96%**, versus
**81.84%** for large and **79.78%** for compact. Compact removes **53.73%** of
the parameters but loses **2.19 macro-F1 points**. Large uses **2.15 times** the
reference parameter count without an overall score improvement.

Large improves Checkmate recall (81.51% versus 80.98%) but loses Nothing recall
(87.89% versus 88.81%). It fixes 4,804 reference errors while breaking 4,954
reference-correct positions: **150 more errors overall**. Check remains the
weakest class for all three models. Keep the reference for the current delivery.

Large has lower selected train loss (0.2847 versus 0.2907) but higher validation
loss (0.4375 versus 0.4325). This is consistent with fitting train better without
improving generalization; it does not isolate the cause. Compact's best epoch
is late (494/500), so its attainable quality is not settled by this run either.

Do not rank training speed from the recorded durations: the reference excludes
its first 100 epochs, and backend conditions differ. Compact and large runners
flagged a changed executable after training; their failed manifests are kept
intact. This comparison separately evaluates their saved models with one frozen
analyzer. Test scores do not decide this ranking; earlier test evaluations exist.

The [100-epoch follow-up](../docs/report/2026-11-07-batch-learning-rate-100epochs.md)
measures batch 128 / LR 0.08 at 25.1% less epoch compute with nearly identical
checkpoint macro F1 versus batch 32 / LR 0.02. The full selected training is in
progress; final convergence is not established.
[Protocol, evidence and limits](../docs/report/2026-11-07-three-models.md).
